# Visual Geometry Group (VGG) Models 2015

## Objective
1. To a gain a better understanding of VGG image classification models introdued in the paper **[VERY DEEP CONVOLUTIONAL NETWORKS FOR LARGE-SCALE IMAGE RECOGNITION](https://arxiv.org/pdf/1409.1556)**

**Note**:
- To look at the entire model's implementation at one place you can have a look at the model's code written here`computer-vision/src/computer_vision/classification/vgg.py`
- To understand image classification thoroughly look at the information provided under the LeNet 1 notebook `computer-vision/src/computer_vision/classification/lenet_1.ipynb`
- To understand dropout layers in depth you can look at [Dropout Layer Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/dropout_layers.ipynb)
- To understand pooling layers in depth you can look at [Pooling Layers Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/pooling_layers.ipynb)
- To understand what batch normalization is in depth you can look at [Batch Normalization Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/batch_normalization.ipynb)
- To understand convolution neural network layer in depth you can look at [Convolution Neural Network Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/convolution_layers.ipynb) (**I advised you to go through this notebook before you go ahead for better understanding of convolution layers**)

## Concepts Introduced in VGG Models
The authors of the paper incorporated 2 concepts into their models and these are<br>
1. Replace convolution filters of large kernel size with stacks of convolution filters of small kernel size
2. To use 1x1 convolution stacks of convolution filters

We will look into them one by one

### 1. Replace convolution filters of large kernel size with stacks of convolution filters of small kernel size
Previously top performing models such as AlexNet on ImageNet, LeNet 5 on MNIST use very large convolution filters of size 11, 7, 5 etc. The authors argue that a **single** convolution filter has very low discriminatory power and it can not completely extract information second it requires large number of parameters for large convolution filters. The authors propose using a stack of smaller convolution filters to capture the same receptive field as this would make the **decision function more discriminative** as multiple alternative convolution layers followed by non linear activation layers will be present and it will **reduce the total number of parameters to capture the same receptive field**.

To replace a **5x5** convolution filter we would need a stack of 2 **3x3** convolution filters and for a **7x7** convolution filter we would need a stack of 3 **3x3** convolution filters.

#### 1.1 How does a stack of smaller convolution filter can capture the same receptive field as that of a single large convolution filter
**Receptive Field** is the size/slice of the input that a convolution filter looks at any given time. For a 7x7 filter the receptive field is 7x7 slice of the input, for a 5x5 filter the receptive field is 5x5 slice of the input and for a 3x3 filter the receptive field is 3x3 slice of the input. A convolution filter performs a convolution operation which is a linear operation and has low descriminatory power. It is because the convolution output is passed through the non linear activation function for it to learn information in a non linear space. What a stack of smaller convolution filter will do is that they will not only capture the same receptive field as that of a large convolution filter they will also stack their low descriminatory power in order to get a strong descriminatory power as the output will transition through multiple non linear activation functions. This will allow the stack to extract more information from the same input

To understand it a bit better we will look at an example.<br>
Let's use a single channeled 9 by 9 tensor where all values apart from the one at the center is zero. 

In [1]:
import torch

# Create a tensor containing zeroes of the shape (1, 1, 9, 9)
x = torch.zeros(size=(1,1,9,9))
# Change the value to 1 for the center pixel
x[0, 0, 4, 4] = 1

print("Original Tensor")
print(x)

Original Tensor
tensor([[[[0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 1., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.]]]])


We will pass the original tensor through a 2D Sum Pooling Layer (2D Average Pooling Layer with the divisor as 1). This will be used in place of a normal convolution layer for simplification. First we will use a large 7x7 convolving pooling filter. We will use a padding of 2 and stride of 1 in order avoid down sampling and preserving the same shape

In [2]:
pool_7x7 = torch.nn.AvgPool2d(kernel_size=7, stride=1, padding=3, divisor_override=1)

result_7x7 = pool_7x7(x)

print("Result of passing the original tensor through a 7x7 2D Sum Pooling Tensor")
print(result_7x7)

Result of passing the original tensor through a 7x7 2D Sum Pooling Tensor
tensor([[[[0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 1., 1., 1., 1., 1., 1., 1., 0.],
          [0., 1., 1., 1., 1., 1., 1., 1., 0.],
          [0., 1., 1., 1., 1., 1., 1., 1., 0.],
          [0., 1., 1., 1., 1., 1., 1., 1., 0.],
          [0., 1., 1., 1., 1., 1., 1., 1., 0.],
          [0., 1., 1., 1., 1., 1., 1., 1., 0.],
          [0., 1., 1., 1., 1., 1., 1., 1., 0.],
          [0., 0., 0., 0., 0., 0., 0., 0., 0.]]]])


Now we will pass the original tensor through a stack 3 of 2D 3x3 Sum Pooling Tensor. Padding of 1 and stride of 1 will be used to preserver the original shape

In [3]:
stack_3x3 = torch.nn.Sequential(
    torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=1, divisor_override=1),
    torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=1, divisor_override=1),
    torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=1, divisor_override=1))

result_3x3 = stack_3x3(x)

print("Result of passing the original tensor through a stack of 3 3x3 2D Sum Pooling Tensor")
print(result_3x3)

Result of passing the original tensor through a stack of 3 3x3 2D Sum Pooling Tensor
tensor([[[[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
          [ 0.,  1.,  3.,  6.,  7.,  6.,  3.,  1.,  0.],
          [ 0.,  3.,  9., 18., 21., 18.,  9.,  3.,  0.],
          [ 0.,  6., 18., 36., 42., 36., 18.,  6.,  0.],
          [ 0.,  7., 21., 42., 49., 42., 21.,  7.,  0.],
          [ 0.,  6., 18., 36., 42., 36., 18.,  6.,  0.],
          [ 0.,  3.,  9., 18., 21., 18.,  9.,  3.,  0.],
          [ 0.,  1.,  3.,  6.,  7.,  6.,  3.,  1.,  0.],
          [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]]]])


We can see that the stack was able to capture the same receptive field as the boundary where values exist in the output is the same for both cases. Second you can see that the stack was able to extract more information from the same receptive field.

#### 1.2 How are we using less parameters when we use a stack of convolution filters when compared to a single large convolution filter
Let's first look at how to calculate the number of trainable parameters in case of a convolution filter. If we are going from $C_{in}$ channels to $C_{out}$ channels in a single convolution function and if the dimensions of a convolution function is given by $(H,W)$ where $H$ is the height of a single convolution kernel and $W$ is it's width. Ignoring the bias term for simplification equation for the total number of trainable parameters is <br>
$$TrainableParameters = C_{in} * C_{out} * H * W$$

If we take the same example as above of 7x7 and 3 3x3 convolution filter we will have <br>
$$TrainableParameters_{7x7} = C_{in} * C_{out} * 7 * 7 => 49 * C_{in} * C_{out}$$
$$TrainableParameters_{3x3} = C_{in} * C_{out} * 3 * 3 => 9 * C_{in} * C_{out}$$
$$3 * TrainableParameters_{3x3} =3 * 9 * C_{in} * C_{out} => 27 * C_{in} * C_{out}$$

We see that a stack of 3 3x3 convolution filters have **45%** less trainable parameters but capture more information from the the same receptive field.

### 2. To use 1x1 convolution stacks of convolution filters
There is a paper called [Network in Network](https://arxiv.org/pdf/1312.4400) which states that convolution filters have low descriminatory power on their own as they are linear functions and in order to extract more information the authors suggest using a multi layered perceptron between two convolution layers to increase the amount of information that can be extracted. They suggest that instead of a multi layered perceptron you can also use a Convolution layer of kernel size 1x1 which will effectly do the samething and save you upon some trainable parameters to train. In VGG paper there are 6 models where one of it's 16 layer model (model C) used this concept by deploying a Convolution layer with 1x1 kernel as the last layer of the convolution blocks.

## Model Architecture and Training Pipeline
The model has been trained on the ImageNet 1k class dataset which contains images of shape $(3,224,224)$.  The paper proposes 6 models (2 11 layered, 1 13 layered, 2 16 layered and 1 19 layered).The models have 2 main blocks the **feature extraction** block and **classifier** block. The general pattern that the feature extraction block is as follows

The feature extraction block as 5 sub blocks of convolution layers (instead of 1 single convolution layer with a large kernel size it contains a stack of smaller kernel size convolution layers). Each convolution block is followed by a Max Pooling Layer that reduces the feature map size by half. Whenever the feature maps are halved the number of channels double in the next block (i.e. every convolution sub block sees the number of input channels double).

All the 6 models have the same classifier block dimensions. The classifier block consists of 3 feedforward layers of the format 4096->4096->1000. Where the last layer is the output layer.

We will understand only one mode **VGG 19 layer model** as others follow the same pattern. Other models (along with their batch normalized version) have also been implemented here `computer-vision/src/computer_vision/classification/vgg.py`. 

### Understanding Feature Extraction Block of VGG 19
The feature extraction block of VGG contains total 5 convolution blocks where each convolution block is followed by a **Max Pooling** layer which halves the size of the feature image.

#### 1. Convolution Block 1
This block takes an input of shape $(N,3,224,224)$ which is an RGB image. The final output is of the shape $(N,64,224,224)$. This block consists of 2 convolution layers where the convolution layers have the same convolution filter dimensions (kernel size = 3, padding=1, stride=1, dilation=1 and a bias for each convolution filter). Only the first layer increases the number of channels from 3 to 64 and the second one keeps this number constant to 64.

The **Max Pooling** layer takes the input $(N,64,224,224)$ and halves it using pooling filters with kernel size 2, stride 2, padding 0 and dilation 1. The output of this layer becomes $(N,64,112,112)$

#### 2. Convolution Block 2
This block follows the same format of increase the number of channels by doubling it in the first layer and keeping the same number of channels throughout the block. It comprises of 2 convolution layers having the same filter dimensions as that of the first convolution block. The first convolution layer increases the number of channels from **64** to **128**.

The **Max Pooling** layer takes the input $(N,128,112,112)$ and halves it using pooling filters with the same filter attributes as the first max pooling layer. The output of this layer becomes $(N,128,56,56)$

#### 3. Convolution Block 3
This block has 4 convolution layers. The dimension for convolution filter for each convolution layer is the same and is the same as that of the previous layers. The first layer increases the channels from **128** to **256** and the remaining three keep the number of feature channels constant.

The **Max Pooling** layer takes the input $(N,256,56,56)$ and halves it using pooling filters. The output of this layer becomes $(N,256,28,28)$

#### 4. Convolution Block 4
This block has the same template as that of convolution block 3. It has 4 convolution layers and the first layer increases the number of channels from **256** to **512** and the remaining three keep the number of feature channel constant.

The **Max Pooling** layer takes the input $(N,512,28,28)$ and halves it using pooling filters. The output of this layer becomes $(N,512,14,14)$

#### 5. Convolution Block 5
This block is the same as **Block 4** and **Block 3**.It has 4 convolution layers and the layers in this model keep the number of channels constant aT **512**.

The **Max Pooling** layer takes the input $(N,512,14,14)$ to $(N,512,7,7)$

### Understanding Classifier Block of VGG 19
This block comprises of 3 fully connected layers where the input to this block is first flattened. The first two fully connected layers use output of dropout layers which have a probability of 50% to drop a neuron. the input goes from $(N, 25088) -> (N,4096) -> (N,4096) -> (N,1000)$

**Rectified Non Linear Activation Function** (ReLU) is used as the non linear activation function through out the model.


Below is the summary of the VGG 19 .

In [4]:
import torch
from torchinfo import summary
from computer_vision.classification.vgg import VGGE


summary(VGGE(1000), input_size=(1, 3, 224, 224), col_names=["input_size", "output_size", "kernel_size", "num_params"])

Layer (type:depth-idx)                   Input Shape               Output Shape              Kernel Shape              Param #
VGGE                                     [1, 3, 224, 224]          [1, 1000]                 --                        --
├─Sequential: 1-1                        [1, 3, 224, 224]          [1, 64, 112, 112]         --                        --
│    └─Conv2d: 2-1                       [1, 3, 224, 224]          [1, 64, 224, 224]         [3, 3]                    1,792
│    └─ReLU: 2-2                         [1, 64, 224, 224]         [1, 64, 224, 224]         --                        --
│    └─Conv2d: 2-3                       [1, 64, 224, 224]         [1, 64, 224, 224]         [3, 3]                    36,928
│    └─ReLU: 2-4                         [1, 64, 224, 224]         [1, 64, 224, 224]         --                        --
│    └─MaxPool2d: 2-5                    [1, 64, 224, 224]         [1, 64, 112, 112]         2                         --
├─Sequential